In [23]:
from functools import partial
import jax
import jax.numpy as jnp
import numpy as np
import queue
import threading


# ---------------------------------------------
# 1) JIT-compiled "replace" and "sample" ops
# ---------------------------------------------
@jax.jit
def replace_cache_jit(loader_state, new_cache):
    """
    Replace the old cache entirely with 'new_cache'.
    Returns an updated loader_state dict.
    """
    return {
        "cache": new_cache,
        "rng":   loader_state["rng"],
    }

@partial(jax.jit, static_argnums=(1,))
def sample_cache_jit(loader_state, batch_size):
    """
    Sample a random minibatch of 'batch_size' from loader_state['cache'].
    Returns (new_loader_state, batch).
    """
    rng = loader_state["rng"]
    rng, subkey = jax.random.split(rng)

    cache = loader_state["cache"]
    cache_size = cache.shape[0]
    idxs = jax.random.randint(subkey, (batch_size,), 0, cache_size)

    batch = cache[idxs, ...]  # fancy indexing
    new_loader_state = {
        "cache": cache,
        "rng":   rng,
    }
    return new_loader_state, batch


# ---------------------------------------------
# 2) SyntheticDataLoader (Double-Buffer style)
# ---------------------------------------------
class SyntheticDataLoader:
    """
    A data loader that:
      - Maintains a single "current" cache (size = cache_size).
      - A background thread produces a *new* full cache in the queue (one big batch).
      - As soon as that big batch is ready, we *swap* out the old cache and replace it.
      - We sample randomly from the current cache for training.

    No partial/rolling updates -- each time we replace, it's a full new dataset.
    """

    def __init__(
        self,
        simulator,
        cache_size=1024,
        data_shape=(28, 28),
        max_queue_size=2,
        store_on_device=True,
        seed=42,
    ):
        """
        :param simulator: a function with signature:
            simulator(rng_key, batch_size, data_shape) -> (new_rng_key, data)
          Must return a JAX or NumPy array of shape (batch_size, *data_shape).
          We'll call it with batch_size = cache_size each time.
        :param cache_size: number of samples to store in the cache (and produce at once).
        :param data_shape: shape of a single data sample.
        :param max_queue_size: queue capacity for "full-caches" that are produced.
        :param store_on_device: if True, store the cache as a JAX array on GPU/TPU
                                (depending on your default JAX device).
        :param seed: base random seed for both simulator RNG and sampling RNG.
        """
        self.simulator     = simulator
        self.cache_size    = cache_size
        self.data_shape    = data_shape
        self.store_on_device = store_on_device

        # Create an initial empty cache on device:
        cache = jnp.zeros((cache_size,) + data_shape, dtype=jnp.float32)

        # We'll store this in a "loader_state" dict
        self.loader_state = {
            "cache": cache,
            "rng":   jax.random.PRNGKey(seed),  # for sampling from the cache
        }

        # Separate RNG for the simulator in the background thread
        self.simulator_rng = jax.random.PRNGKey(seed + 10_000)

        # A queue where each entry is a *full new cache* of shape (cache_size, ...).
        # We'll push from the background thread, pull in the main thread.
        self.queue = queue.Queue(maxsize=max_queue_size)

        # Create & start background thread
        self._stop_event = threading.Event()
        self._thread = threading.Thread(
            target=self._producer_loop,
            daemon=True
        )
        self._thread.start()

    def _producer_loop(self):
        """Background thread that produces a *full new cache* (cache_size) each iteration."""
        while not self._stop_event.is_set():
            # Produce a single batch of size = cache_size
            self.simulator_rng, new_data = self.simulator(
                self.simulator_rng,
                self.cache_size,
                self.data_shape
            )
            # Convert to NumPy on CPU if it's a jax.Array
            new_data = np.array(new_data, copy=False)

            # Put it in the queue (blocks if full)
            self.queue.put(new_data)

    def init_cache(self):
        """
        Optionally fill the cache immediately (blocking) before training
        so you start with a "warm" cache.
        """
        self.simulator_rng, big_batch = self.simulator(
            self.simulator_rng,
            self.cache_size,
            self.data_shape
        )
        big_batch_jax = jnp.array(big_batch)  # ensure a jax Array
        self.loader_state = replace_cache_jit(self.loader_state, big_batch_jax)

    def maybe_replace_cache(self):
        """
        If there's a *complete new cache* in the queue, replace the old one.
        Otherwise, do nothing.
        """
        if not self.queue.empty():
            # Pull the new data
            new_batch_np = self.queue.get()
            # Convert to jax array
            new_batch_jax = jnp.array(new_batch_np)
            # Swap out old cache for new
            self.loader_state = replace_cache_jit(self.loader_state, new_batch_jax)

    def get_batch_from_cache(self, batch_size):
        """
        1) Possibly replace the entire cache if the background thread has one ready.
        2) Sample from the current cache (JIT-compiled).
        """
        # Step 1: see if there's a new cache in the queue
        self.maybe_replace_cache()

        # Step 2: sample
        self.loader_state, batch = sample_cache_jit(self.loader_state, batch_size)
        return batch

    def stop(self):
        """Stop the background producer."""
        self._stop_event.set()
        self._thread.join()


# ---------------------------------------------
# 3) Example simulator function
# ---------------------------------------------
@partial(jax.jit, static_argnums=(1,2))
def my_simulator(rng_key, batch_size, data_shape):
    """
    Just returns random normal data. You can replace
    with your own synthetic generation code.
    """
    rng_key, subkey = jax.random.split(rng_key)
    data = jax.random.normal(subkey, shape=(batch_size,) + data_shape)
    return rng_key, data


# ---------------------------------------------
# 4) Example usage
# ---------------------------------------------
def main():
    # Create a loader that uses the "my_simulator" function,
    # produces entire 10_000-sample caches, and swaps them out in full.
    loader = SyntheticDataLoader(
        simulator=my_simulator,
        cache_size=10_000,
        data_shape=(28, 28),
        store_on_device=True,
        seed=42
    )

    # Optionally fill the cache once right away
    loader.init_cache()

    # Example trivial train loop
    @jax.jit
    def train_step(params, x):
        # Just sum-of-squares loss for demonstration
        loss = jnp.mean(x * x)
        return params, loss

    params = jnp.array(0.0)  # dummy
    for step in range(5):
        # Get a mini-batch of size 32 from the current (possibly just swapped) cache
        batch = loader.get_batch_from_cache(batch_size=32)
        params, loss = train_step(params, batch)
        print(f"Step {step}, loss = {loss}")

    loader.stop()


In [34]:
loader = SyntheticDataLoader(
        simulator=my_simulator,
        cache_size=10_000,
        data_shape=(10,),
        store_on_device=False,
        seed=42
    )
loader.init_cache()

In [51]:
%%timeit
my_simulator(jax.random.PRNGKey(42), 10_000, (10,))

479 μs ± 5.46 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [37]:
%%timeit
loader.get_batch_from_cache(batch_size=1024).block_until_ready()

759 μs ± 17.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [45]:
my_simulator

<PjitFunction of <function my_simulator at 0x7f9a1010b600>>

In [22]:
loader.get_batch_from_cache(batch_size=32)

Array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0.